# A small from-scratch CNN on EuroSAT_RGB (~96k parameters)

Same task as `train_resnet18.ipynb` — **`blanchon/EuroSAT_RGB`** from the Hugging Face Hub,
27,000 Sentinel-2 satellite patches, 10 land-use classes, 64x64 RGB — but a completely different
model: a compact CNN trained **from scratch**, with **~96k parameters** (about 0.8% of ResNet-18's
11.7M).

The point is the opposite of the transfer-learning notebook: no pretrained weights, no 224x224
upscaling, and a network small enough to train on a laptop CPU in a few minutes per epoch. That
makes it a cheap baseline for benchmarks or weight-space experiments.

**What this notebook covers**

1. Loading the dataset from the Hub (official train/validation/test splits)
2. Defining a ~96k-parameter CNN and checking the parameter budget
3. Training from scratch at native 64x64 with cosine LR decay
4. Loss/accuracy curves, confusion matrix, and sample predictions
5. Checkpoint save/load, 96-d feature extraction, and a flattened weight vector

**Requirements:** `torch`, `torchvision`, `datasets`, `matplotlib`, `scikit-learn`

> **Note on the data:** same as the ResNet notebook, the dataset comes from the Hub, so
> `data/cifar-10-python.tar.gz` in this repo is not used. Because the model is trained from
> scratch, there is no reason to upscale: everything runs at the native 64x64.

In [ ]:
# If something is missing:
# %pip install torch torchvision datasets matplotlib scikit-learn

## 1. Setup and configuration

All data comes from `load_dataset("blanchon/EuroSAT_RGB")`, which returns a `DatasetDict` with
`train` / `validation` / `test` splits already separated. We use them as-is, so the test accuracy is
genuinely held out.

Starting from random weights needs more epochs than fine-tuning did. `EPOCHS = 20` is a reasonable
default; drop to `EPOCHS = 2`, `TRAIN_FRACTION = 0.05` to check that the notebook runs end to end.

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, models
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# ----------------------------- config -----------------------------
SEED            = 42
HF_DATASET      = "blanchon/EuroSAT_RGB"   # 64x64 RGB, 10 classes
IMG_SIZE        = 64                # native resolution; no upscaling needed from scratch
BATCH_SIZE      = 64
NUM_WORKERS     = 0                 # Windows/Jupyter: keep 0
TRAIN_FRACTION  = 1.0               # e.g. 0.05 for a quick smoke test
EPOCHS          = 20
LR              = 1e-3              # one learning rate for the whole (small) network
WEIGHT_DECAY    = 5e-4
LABEL_SMOOTHING = 0.1
DROPOUT         = 0.2               # before the classifier
CKPT_PATH       = Path("../checkpoints/smallcnn_eurosat.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = device.type == "cuda"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("device:", device)
print("torch:", torch.__version__)

## 2. Data and augmentation

Augmentation matters much more when training from scratch than when fine-tuning a pretrained
backbone. Training uses a random resized crop, horizontal **and** vertical flips (overhead imagery
has no canonical orientation, so a vertical flip is label-preserving here) and a bit of colour
jitter. Validation/test use a plain resize only.

The same small wrapper turns each Hugging Face split into a `torch.utils.data.Dataset`, so the
regular `DataLoader` can apply the per-split transforms.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class HFImageDataset(Dataset):
    '''Wrap a Hugging Face image split as a torch Dataset (image, label).'''

    def __init__(self, split, transform):
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        item = self.split[int(idx)]
        image = item["image"].convert("RGB")
        return self.transform(image), item["label"]


raw = load_dataset(HF_DATASET)
classes = raw["train"].features["label"].names
num_classes = len(classes)

train_full = HFImageDataset(raw["train"], train_tf)
val_ds     = HFImageDataset(raw["validation"], eval_tf)
test_ds    = HFImageDataset(raw["test"], eval_tf)

# optional quick smoke test: subsample the training split only
if TRAIN_FRACTION < 1.0:
    g = torch.Generator().manual_seed(SEED)
    n_keep = max(1, int(len(train_full) * TRAIN_FRACTION))
    keep = torch.randperm(len(train_full), generator=g)[:n_keep].tolist()
    train_ds = Subset(train_full, keep)
else:
    train_ds = train_full

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

print(f"classes ({num_classes}): {classes}")
print(f"train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")

In [ ]:
def denormalize(t):
    t = t.clone()
    for c, (m, s) in enumerate(zip(MEAN, STD)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

images, labels = next(iter(train_loader))
images, labels = images[:16], labels[:16]

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(classes[int(lab)], fontsize=8)
    ax.axis("off")
fig.suptitle("Training samples (with augmentation)", y=1.02)
plt.tight_layout()
plt.show()

## 3. Model: a ~96k-parameter CNN

Four convolutional stages — `3 -> 24 -> 48 -> 64 -> 96` channels — each a `3x3` conv followed by
BatchNorm, ReLU and `2x2` max-pool. On a 64x64 input that leaves a `4x4x96` feature map, which is
globally average-pooled (so the classifier is only `96 x 10` weights), followed by dropout and a
linear layer.

Design notes:

* **Single conv per stage.** Two convs per stage would roughly double the parameters; with a ~100k
  budget one conv per stage is the better trade.
* **Global average pooling** instead of a large flatten + linear head, which keeps the head at
  ~1k parameters and makes the model resolution-robust.
* **BatchNorm** is what makes from-scratch training at this size stable with a plain AdamW setup.

The cell prints the parameter count, and compares it against a freshly built (not pretrained)
ResNet-18 to show the scale of the difference.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

class SmallCNN(nn.Module):
    '''Four conv stages + global average pooling + linear head (~96k parameters).'''

    def __init__(self, num_classes, channels=(24, 48, 64, 96), dropout=DROPOUT):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.features = nn.Sequential(
            conv_block(3, c1),      # 64 -> 32
            conv_block(c1, c2),     # 32 -> 16
            conv_block(c2, c3),     # 16 -> 8
            conv_block(c3, c4),     # 8  -> 4
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(c4, num_classes),
        )

    def forward_features(self, x):
        '''Penultimate 96-d descriptor (global average pooled conv features).'''
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


model = SmallCNN(num_classes).to(device)

total = sum(p.numel() for p in model.parameters())
resnet = models.resnet18(weights=None)
resnet_total = sum(p.numel() for p in resnet.parameters())

print(f"small cnn : {total:,} parameters ({total/1e3:.1f}k)")
print(f"resnet-18 : {resnet_total:,} parameters ({resnet_total/1e6:.2f}M)")
print(f"ratio     : {total/resnet_total:.3%} of ResNet-18")

assert 80_000 <= total <= 120_000, "parameter budget drifted away from ~100k"
print("\nlayers:")
for name, module in model.features.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  stage {name}: {n:>6,} params, {module[0].out_channels} channels")
print(f"  head     : {sum(p.numel() for p in model.classifier.parameters()):>6,} params")

## 4. Loss, optimiser and training loop

There is no frozen/unfrozen schedule here — every weight is learned from scratch, so a single
AdamW setup for the whole run is enough:

* **One learning rate**, decayed with a cosine schedule over `EPOCHS`.
* **No weight decay on BatchNorm weights and biases** — only on conv/linear weights.
* **Label smoothing** (`0.1`) as a cheap regulariser on a small dataset.
* **AMP** — mixed precision is enabled automatically on CUDA and skipped on CPU.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def make_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY):
    '''Decay conv/linear weights, leave BatchNorm weights and biases alone.'''
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    groups = [{"params": decay, "weight_decay": weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=lr)

optimizer = make_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

print(f"optimiser groups: {len(optimizer.param_groups)} | amp: {use_amp}")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()

    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        seen += x.size(0)

    return loss_sum / seen, correct / seen

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, seen = 0.0, 0, 0
    preds, targets = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(device.type, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        seen += x.size(0)
        preds.append(pred.cpu())
        targets.append(y.cpu())

    return (loss_sum / seen, correct / seen,
            torch.cat(preds).numpy(), torch.cat(targets).numpy())

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_val_acc = -1.0
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)

    cur_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    history["lr"].append(cur_lr)

    print(f"epoch {epoch + 1:02d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.3f} | "
          f"lr {cur_lr:.2e} | {time.time() - t0:.0f}s")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({"model": model.state_dict(), "classes": classes,
                    "img_size": IMG_SIZE, "channels": (24, 48, 64, 96),
                    "val_acc": va_acc}, CKPT_PATH)
        print(f"  -> new best saved ({va_acc:.3f})")

print(f"best val accuracy: {best_val_acc:.3f}")

## 5. Training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
best_epoch = int(np.argmax(history["val_acc"])) + 1

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs, history["val_loss"], marker="o", label="val")
axes[0].set(title="Loss", xlabel="epoch", ylabel="cross-entropy")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs, history["val_acc"], marker="o", label="val")
axes[1].axvline(best_epoch, color="grey", ls="--", alpha=0.6, label=f"best ({best_epoch})")
axes[1].set(title="Accuracy", xlabel="epoch", ylabel="accuracy")
axes[1].legend()

axes[2].plot(epochs, history["lr"], marker="o", color="tab:green")
axes[2].set(title="Learning rate", xlabel="epoch", ylabel="lr")
axes[2].set_yscale("log")

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("Small CNN (~96k params) from scratch on EuroSAT_RGB", y=1.03)
plt.tight_layout()
plt.show()

## 6. Evaluate the best checkpoint

This is the official EuroSAT `test` split, which the model never saw during training or for model
selection (that used the `validation` split). This is a real held-out test number — expect it to
sit well below the fine-tuned ResNet-18, which is the whole point of having both notebooks.

In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model"])
model.eval()
print(f"loaded best checkpoint (val acc {checkpoint['val_acc']:.3f})")

test_loss, test_acc, y_pred, y_true = evaluate(
    model, test_loader, nn.CrossEntropyLoss())
print(f"test: loss {test_loss:.4f} | accuracy {test_acc:.3f}")

print("\n" + classification_report(y_true, y_pred, labels=range(num_classes),
                                   target_names=classes, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set(title="Confusion matrix (test)", xlabel="predicted", ylabel="true")
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(classes, fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="row-normalised")
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def show_predictions(model, loader, n=12):
    model.eval()
    images, labels = next(iter(loader))
    images = images[:n]
    preds = model(images.to(device)).argmax(1).cpu()

    cols = 6
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

    for ax, img, true, pred in zip(axes.ravel(), images, labels, preds):
        ax.imshow(denormalize(img).permute(1, 2, 0))
        ok = int(true) == int(pred)
        ax.set_title(f"true: {classes[int(true)]}\npred: {classes[int(pred)]}",
                     fontsize=8, color="green" if ok else "crimson")
        ax.axis("off")

    for ax in axes.ravel()[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, test_loader)

## 7. Going further

**Reload the model elsewhere** — the last cell defines `load_model(path)`, which rebuilds the
architecture and restores the saved weights.

**96-d penultimate features** (linear probes, clustering, weight-space experiments)

```python
features, targets = extract_features(model, test_loader)   # (N, 96), (N,)
```

**Tips**

* CPU only / quick check? `EPOCHS = 2`, `TRAIN_FRACTION = 0.05` first.
* Underfitting? Widen the first stage only — `channels=(32, 48, 64, 96)` in `SmallCNN` is ~99k
  params and still passes the budget check. Or train longer, or raise `LR` slightly.
* Want more capacity? `channels=(32, 64, 96, 128)` is ~187k; the `assert` in the model cell stops
  you once you leave the ~100k budget, so raise it deliberately if that is what you want.
* Overfitting? Raise `WEIGHT_DECAY` / `DROPOUT`, or add augmentation.
* Class imbalance? Use `WeightedRandomSampler` or pass class weights to `CrossEntropyLoss`.
* To resume training, also save `optimizer.state_dict()`, `scheduler.state_dict()` and the epoch.


In [ ]:
# 96-d penultimate features (global average pooled conv features)
@torch.no_grad()
def extract_features(model, loader):
    '''Penultimate features: (N, 96), (N,).'''
    model.eval()
    feats, ys = [], []
    for x, y in loader:
        feats.append(model.forward_features(x.to(device)).cpu())
        ys.append(y)
    return torch.cat(feats), torch.cat(ys)

features, targets = extract_features(model, test_loader)
print("features:", tuple(features.shape), "| targets:", tuple(targets.shape))

In [ ]:
# Weight-space view: flatten every parameter into a single vector
def state_dict_to_vector(model):
    return torch.cat([p.detach().flatten().cpu() for p in model.parameters()])

w = state_dict_to_vector(model)
print("weight vector:", tuple(w.shape))

In [ ]:
def load_model(path, device=device):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    m = SmallCNN(len(ckpt["classes"]), channels=ckpt.get("channels", (24, 48, 64, 96)))
    m.load_state_dict(ckpt["model"])
    return m.to(device).eval(), ckpt["classes"]

# model2, classes2 = load_model(CKPT_PATH)